## 전처리

In [2]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import sys
from scipy.stats import spearmanr

In [3]:
# 1. scan_csv를 사용하여 지연 평가(Lazy) 방식으로 연결해
# 인코딩이 CP949일 가능성이 높으니 에러가 나면 encoding="cp949"를 추가해줘
try:
    # 데이터 경로를 실제 파일명으로 수정해!
    pos_lazy = pl.scan_csv("/Users/hajiyoon/dataset_seveneleven/B2_POS_SALE_H1.csv") 
    
    # 2. head(10)을 호출하고 collect()로 실행해
    # 지연 평가 방식이라 collect()를 해야 실제 결과가 출력돼
    pos_head = pos_lazy.head(10).collect()
    
    print("--- POS 데이터 상위 10개 행 ---")
    print(pos_head)

except Exception as e:
    print(f"파일을 읽는 중 에러가 발생했어: {e}")
    print("팁: 인코딩 문제라면 scan_csv(..., encoding='cp949')를 시도해봐!")

--- POS 데이터 상위 10개 행 ---
shape: (10, 8)
┌───────────┬───────────┬────────────┬────────┬──────────┬───────────┬──────────┬──────────┐
│ SALE_DATE ┆ SALE_TIME ┆ STORE_CODE ┆ POS_NO ┆ TRADE_NO ┆ ITEM_CODE ┆ SALE_QTY ┆ SALE_AMT │
│ ---       ┆ ---       ┆ ---        ┆ ---    ┆ ---      ┆ ---       ┆ ---      ┆ ---      │
│ i64       ┆ i64       ┆ i64        ┆ i64    ┆ i64      ┆ i64       ┆ i64      ┆ str      │
╞═══════════╪═══════════╪════════════╪════════╪══════════╪═══════════╪══════════╪══════════╡
│ 20250523  ┆ 84531     ┆ 64139      ┆ 2      ┆ 26651    ┆ 314952    ┆ 1        ┆ 500      │
│ 20250523  ┆ 105724    ┆ 52657      ┆ 1      ┆ 48727    ┆ 314954    ┆ 10       ┆ 45,000   │
│ 20250523  ┆ 105724    ┆ 52657      ┆ 1      ┆ 48727    ┆ 314952    ┆ 10       ┆ 5,000    │
│ 20250523  ┆ 84531     ┆ 64139      ┆ 2      ┆ 26651    ┆ 314983    ┆ 1        ┆ 4,500    │
│ 20250523  ┆ 83823     ┆ 50134      ┆ 1      ┆ 67971    ┆ 314988    ┆ 1        ┆ 4,500    │
│ 20250523  ┆ 113903    ┆ 6807

In [4]:
# 데이터 형변환 및 컬럼명 변경
pos_processed_lazy = pos_lazy.select([
    pl.col("SALE_DATE").cast(pl.Utf8).alias("판매일자"),
    
    # 6자리가 되도록 앞에 0을 채움 (예: 84531 -> "084531")
    pl.col("SALE_TIME").cast(pl.Utf8).str.zfill(6).alias("판매시간"),
    
    pl.col("STORE_CODE").cast(pl.Utf8).alias("점포코드"),
    
    # 2자리가 되도록 앞에 0을 채움 (예: 2 -> "02")
    pl.col("POS_NO").cast(pl.Utf8).str.zfill(2).alias("POS번호"),
    
    pl.col("TRADE_NO").cast(pl.Utf8).alias("거래번호"),
    
    pl.col("ITEM_CODE").cast(pl.Utf8).alias("상품코드"),
    
    # 수량은 기존 i64 유지, 컬럼명만 변경
    pl.col("SALE_QTY").cast(pl.Int64).alias("판매수량"),
    
    # 금액은 쉼표 제거 후 정수형(i64) 변환
    pl.col("SALE_AMT").str.replace_all(",", "").cast(pl.Int64).alias("판매금액")
])

# 3. 전처리 결과 상위 5개 행 확인
pos_processed_df = pos_processed_lazy.head(5).collect()
print(pos_processed_df)

shape: (5, 8)
┌──────────┬──────────┬──────────┬─────────┬──────────┬──────────┬──────────┬──────────┐
│ 판매일자 ┆ 판매시간 ┆ 점포코드 ┆ POS번호 ┆ 거래번호 ┆ 상품코드 ┆ 판매수량 ┆ 판매금액 │
│ ---      ┆ ---      ┆ ---      ┆ ---     ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ str      ┆ str      ┆ str      ┆ str     ┆ str      ┆ str      ┆ i64      ┆ i64      │
╞══════════╪══════════╪══════════╪═════════╪══════════╪══════════╪══════════╪══════════╡
│ 20250523 ┆ 084531   ┆ 64139    ┆ 02      ┆ 26651    ┆ 314952   ┆ 1        ┆ 500      │
│ 20250523 ┆ 105724   ┆ 52657    ┆ 01      ┆ 48727    ┆ 314954   ┆ 10       ┆ 45000    │
│ 20250523 ┆ 105724   ┆ 52657    ┆ 01      ┆ 48727    ┆ 314952   ┆ 10       ┆ 5000     │
│ 20250523 ┆ 084531   ┆ 64139    ┆ 02      ┆ 26651    ┆ 314983   ┆ 1        ┆ 4500     │
│ 20250523 ┆ 083823   ┆ 50134    ┆ 01      ┆ 67971    ┆ 314988   ┆ 1        ┆ 4500     │
└──────────┴──────────┴──────────┴─────────┴──────────┴──────────┴──────────┴──────────┘


In [5]:
# Parquet 형식으로 저장 (압축 및 스키마 보존)
pos_processed_lazy.collect().write_parquet("B2_POS_SALE.parquet")

ComputeError: could not parse `"-1,000"` as dtype `i64` at column 'SALE_QTY' (column number 7)

The current offset in the file is 356780 bytes.

You might want to try:
- increasing `infer_schema_length` (e.g. `infer_schema_length=10000`),
- specifying correct dtype with the `schema_overrides` argument
- setting `ignore_errors` to `True`,
- adding `"-1,000"` to the `null_values` list.

Original error: ```invalid primitive value found during CSV parsing```

## 1. 데이터 스캐닝 및 기초 프로파일링

목적: 대용량 POS 데이터의 물리적 크기를 확인하고, 스키마 및 결측치 구조를 파악하여 데이터의 정합성을 검증한다.
데이터 : pos 데이터